#### Data Cleaning and Processing

#### Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

#### 1. Load Data

In [2]:
# Load the file
df = pd.read_csv('inventory_data.csv')

# Convert date columns properly
date_columns = ['last_received_date', 'last_distributed_date', 'expiry_date']
for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors='coerce')

print("Shape:", df.shape)
print("\nData types & non-null counts:")
print(df.info())

Shape: (1500, 16)

Data types & non-null counts:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   warehouse_id             1500 non-null   object        
 1   country                  1500 non-null   object        
 2   item_code                1500 non-null   object        
 3   item_type                1500 non-null   object        
 4   batch_number             1500 non-null   object        
 5   stock_level_current      1500 non-null   int64         
 6   stock_level_initial      1500 non-null   int64         
 7   unit                     1500 non-null   object        
 8   last_received_date       1500 non-null   datetime64[ns]
 9   last_distributed_date    1500 non-null   datetime64[ns]
 10  expiry_date              1500 non-null   datetime64[ns]
 11  condition_flag           1500 non-null   objec

In [3]:
print("\nMissing values:")
print(df.isna().sum())


Missing values:
warehouse_id               0
country                    0
item_code                  0
item_type                  0
batch_number               0
stock_level_current        0
stock_level_initial        0
unit                       0
last_received_date         0
last_distributed_date      0
expiry_date                0
condition_flag             0
donor_compliance_status    0
waybill_reference          0
is_expired                 0
days_to_expiry             0
dtype: int64


#### 2. Basic quality cleaning

##### Identify rows with any invalid date

In [4]:
invalid_date_mask = df[date_columns].isna().any(axis=1)
print(f"Rows with invalid dates: {invalid_date_mask.sum()} ({invalid_date_mask.mean():.2%})")

# Remove rows with completely broken dates
df = df[~invalid_date_mask].copy()

Rows with invalid dates: 0 (0.00%)


##### Remove exact duplicates

In [5]:
before_dedup = len(df)
df = df.drop_duplicates()
print(f"Duplicates removed: {before_dedup - len(df)} rows")

# Sort for operational readability
df = df.sort_values(['country', 'warehouse_id', 'expiry_date']).reset_index(drop=True)

Duplicates removed: 0 rows


#### 3. Stock & expiry consistency fixes

In [6]:
# Define reference date
REF_DATE = datetime(2025, 6, 1)

# Never allow negative stock (common ERP validation rule)
df['stock_level_current'] = df['stock_level_current'].clip(lower=0)

# Re-compute expiry status (robust)
df['is_expired'] = df['expiry_date'] < REF_DATE

# Days until / since expiry
df['days_to_expiry'] = (df['expiry_date'] - REF_DATE).dt.days

print("Expiry status after cleaning:")
print(df['is_expired'].value_counts(normalize=True).mul(100).round(1).to_frame('percentage (%)'))

Expiry status after cleaning:
            percentage (%)
is_expired                
False                 68.4
True                  31.6


#### 4. Create humanitarian-relevant risk & urgency indicators

In [7]:
# Expiry urgency tier
conditions = [
    df['days_to_expiry'] <= 0,
    df['days_to_expiry'] <= 30,
    df['days_to_expiry'] <= 90,
    df['days_to_expiry'] <= 180
]
choices = ['Expired', 'Immediate Action', 'High Priority', 'Medium Priority']
df['expiry_urgency_tier'] = np.select(conditions, choices, default='Low Priority')

# Composite risk score (0–100)
# Higher = needs more attention
df['risk_score'] = 0.0

# Expiry contribution (max 50)
df['risk_score'] += np.where(df['is_expired'], 50,
                             np.where(df['days_to_expiry'] <= 30, 40,
                                      np.where(df['days_to_expiry'] <= 90, 25, 0)))

# Low / zero stock contribution (max 30)
df['risk_score'] += np.where(df['stock_level_current'] == 0, 30,
                             np.where(df['stock_level_current'] <= 50, 20, 0))

# Compliance contribution (max 20)
compliance_map = {
    'Compliant': 0,
    'Pending Audit': 8,
    'Non-Compliant - Access Blocked': 15,
    'Non-Compliant - Documentation': 12
}
df['risk_score'] += df['donor_compliance_status'].map(compliance_map).fillna(5)

df['risk_score'] = df['risk_score'].clip(upper=100).round(1)

print("Risk score distribution:")
print(df['risk_score'].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]).round(1))

Risk score distribution:
count    1500.0
mean       24.7
std        23.4
min         0.0
25%         0.0
50%        15.0
75%        50.0
90%        58.0
95%        62.0
max        65.0
Name: risk_score, dtype: float64


In [8]:
print("\nExpiry urgency distribution:")
print(df['expiry_urgency_tier'].value_counts(normalize=True).mul(100).round(1).to_frame('percentage (%)'))


Expiry urgency distribution:
                     percentage (%)
expiry_urgency_tier                
Low Priority                   39.9
Expired                        31.7
Medium Priority                14.3
High Priority                   9.2
Immediate Action                4.8


#### 5. Supply chain proxy metrics

In [9]:
# Approximate days on hand
df['approx_daily_consumption'] = df['stock_level_initial'] / 180.0  
df['approx_daily_consumption'] = df['approx_daily_consumption'].replace(0, 0.1)

df['days_on_hand'] = df['stock_level_current'] / df['approx_daily_consumption']
df['days_on_hand'] = df['days_on_hand'].clip(upper=365).round(0).astype(int)

# Simple stock status categories
stock_conditions = [
    df['stock_level_current'] == 0,
    df['stock_level_current'] <= 50,
    df['stock_level_current'] <= 200
]
stock_choices = ['Stockout', 'Critical Low', 'Low']
df['stock_status'] = np.select(stock_conditions, stock_choices, default='Adequate')

print("Days on hand summary:")
print(df['days_on_hand'].describe(percentiles=[0.25, 0.5, 0.75, 0.9]).round(0))

print("\nStock status distribution:")
print(df['stock_status'].value_counts(normalize=True).mul(100).round(1).to_frame('percentage (%)'))

Days on hand summary:
count    1500.0
mean      118.0
std        37.0
min        36.0
25%        95.0
50%       125.0
75%       148.0
90%       161.0
max       171.0
Name: days_on_hand, dtype: float64

Stock status distribution:
              percentage (%)
stock_status                
Adequate                99.3
Low                      0.7


##### 6. Final column ordering & export

In [10]:
# Logical column order for downstream use
final_columns = [
    'country', 'warehouse_id',
    'item_type', 'item_code', 'batch_number',
    'stock_level_initial', 'stock_level_current', 'unit',
    'stock_status', 'days_on_hand',
    'last_received_date', 'last_distributed_date',
    'expiry_date', 'days_to_expiry', 'is_expired', 'expiry_urgency_tier',
    'condition_flag', 'donor_compliance_status', 'risk_score',
    'waybill_reference'
]

# Keep only columns that exist
final_columns = [c for c in final_columns if c in df.columns]
df = df[final_columns]

print("Final shape:", df.shape)
print("Final columns:", list(df.columns))

# Export enriched & cleaned version
df.to_csv('cleaned_enriched_inventory.csv', index=False)
print("\nSaved: cleaned_enriched_inventory.csv")

Final shape: (1500, 20)
Final columns: ['country', 'warehouse_id', 'item_type', 'item_code', 'batch_number', 'stock_level_initial', 'stock_level_current', 'unit', 'stock_status', 'days_on_hand', 'last_received_date', 'last_distributed_date', 'expiry_date', 'days_to_expiry', 'is_expired', 'expiry_urgency_tier', 'condition_flag', 'donor_compliance_status', 'risk_score', 'waybill_reference']

Saved: cleaned_enriched_inventory.csv


In [11]:
df.head()

,country,warehouse_id,item_type,item_code,batch_number,stock_level_initial,stock_level_current,unit,stock_status,days_on_hand,last_received_date,last_distributed_date,expiry_date,days_to_expiry,is_expired,expiry_urgency_tier,condition_flag,donor_compliance_status,risk_score,waybill_reference
0,Afghanistan,WH-1164-AFG,Medical Supplies,MS-003,BATCH-92797,3812,2664,bottles,Adequate,126,2025-02-25,2025-02-26,2026-12-31,578,False,Low Priority,Good,Pending Audit,8.0,WB-929149
1,Afghanistan,WH-1166-AFG,Hygiene Kits,HK-001,BATCH-47875,3126,2294,kits,Adequate,132,2024-06-26,2024-11-30,2025-02-07,-114,True,Expired,Expired,Non-Compliant - Documentation,62.0,WB-928602
2,Afghanistan,WH-1212-AFG,Medical Supplies,MS-002,BATCH-85506,999,845,bottles,Adequate,152,2024-12-10,2025-05-06,2026-11-14,531,False,Low Priority,Good,Compliant,0.0,WB-988047
3,Afghanistan,WH-1293-AFG,Medical Supplies,MS-002,BATCH-10910,3307,2357,kg,Adequate,128,2024-07-19,2024-08-17,2025-10-26,147,False,Medium Priority,Good,Pending Audit,8.0,WB-679214
4,Afghanistan,WH-1295-AFG,Medical Supplies,MS-003,BATCH-81511,3966,2273,kg,Adequate,103,2025-01-20,2025-03-31,2026-10-09,495,False,Low Priority,Good,Compliant,0.0,WB-844015
